# Forget-MI LoKU — Kaggle Notebook> 🎯 **Mục đích**: Chạy LoKU trên Kaggle để đo song song với baseline `run_kaggle_baseline.ipynb` trên cùng GPU T4 — đảm bảo so sánh FAIR cho thesis Chương 4.## Cấu trúc| Cell | Mục đích ||---|---|| 1 | Setup Kaggle env: clone repo, install deps, restore CSV || 2 | Auto-detect Kaggle dataset paths + verify || 3 | Define helpers (run_multiseed, aggregate_summary) || 4a/4b/4c | Train multi-seed 3% / 6% / 10% (4b/4c có RUN flag, mặc định OFF) || 5 | Bảng tổng hợp cross-forget% || 6 | Push results lên GitHub |## Setup Kaggle 1 lần1. **Datasets** (Sidebar → + Add data):   - `forget-mi-data` (chứa `data/metadata` + `data/img_data`)   - `forget-mi-models` (chứa `base_model/` + `retrained_model/`)2. **Secrets** (Add-ons → Secrets): `GITHUB_TOKEN`, `GIT_EMAIL`, `GIT_NAME`3. **GPU**: Settings → Accelerator → GPU T4 x2## So với baseline| | Baseline (run_kaggle_baseline.ipynb) | LoKU (notebook này) ||---|---|---|| Script | `forgetmi_partial.py` | `forgetmi_loku.py` || Config | `config_baseline_kaggle.yaml` | `config_loku_kaggle.yaml` || Trainable | 113M (full FT) | ~3M (LoRA + FILA, ~2.6%) || Epochs | 30 | 8 || Time/seed | ~3h | ~15-20 phút || Checkpoint | ~450 MB | ~12 MB || CSV path | `/kaggle/working/results_summary.csv` | `/kaggle/working/unlearning_output/results_summary.csv` || Repo CSV | `results_summary_kaggle.csv` | `results_summary_loku_kaggle.csv` |

In [ ]:
# ====================================# CELL 1: Setup Kaggle env (clone repo + install deps + restore CSV)# ====================================import os, sys, subprocess, shutilWORK_DIR = "/kaggle/working"REPO_URL = "https://github.com/nhnhu146/Forget-MI-LoKU.git"REPO_NAME = "Forget-MI-LoKU"REPO_DIR = f"{WORK_DIR}/{REPO_NAME}"os.chdir(WORK_DIR)# 1. Clone hoặc pull repoif not os.path.exists(REPO_DIR):    print(f"🔽 Clone {REPO_URL}")    subprocess.run(['git', 'clone', REPO_URL], check=True)else:    print(f"🔄 Pull latest")    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin'], check=True)    subprocess.run(['git', '-C', REPO_DIR, 'reset', '--hard', 'origin/master'], check=True)os.chdir(REPO_DIR)print(f"📂 CWD: {os.getcwd()}")subprocess.run(['git', 'log', '--oneline', '-1'])# 2. ⭐ RESTORE CSV từ repo về /kaggle/working/unlearning_output/ (nếu Cell 6 trước đó đã push)csv_in_repo = "experiments/results_summary_loku_kaggle.csv"csv_target = "/kaggle/working/unlearning_output/results_summary.csv"os.makedirs(os.path.dirname(csv_target), exist_ok=True)if os.path.exists(csv_in_repo) and not os.path.exists(csv_target):    shutil.copy(csv_in_repo, csv_target)    n = sum(1 for _ in open(csv_target)) - 1    print(f"♻️  Restored CSV từ repo: {csv_target} ({n} rows)")elif os.path.exists(csv_target):    n = sum(1 for _ in open(csv_target)) - 1    print(f"📊 CSV hiện có: {csv_target} ({n} rows)")else:    print(f"📊 CSV chưa có — chạy từ đầu")# 3. Install depsprint("\n📦 Installing deps...")# FORCE-INSTALL pinned versions (Kaggle pre-installs newer transformers# which breaks ImageTextModel due to BertAttention API changes in 4.45+)print("📦 Force-install pinned deps (transformers 4.38.0 + peft 0.10.0)...")get_ipython().system('pip install -q pydicom scikit-image pyyaml')get_ipython().system('pip install -q --force-reinstall --no-deps \n  "transformers==4.38.0" "peft==0.10.0" "accelerate==0.27.0"')print("✅ Pinned deps installed.")# 4. Check GPUimport torchif torch.cuda.is_available():    print(f"\n🟢 GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")else:    print("\n🔴 KHÔNG CÓ GPU — LoKU sẽ KHÔNG xong trong 12h Kaggle limit!")    print("   → Settings → Accelerator → GPU T4 x2 → Save → Restart Session")print("\n✅ Setup complete")

In [ ]:
# ====================================# CELL 2: Auto-detect Kaggle dataset paths# ====================================import os, globdef _find_kaggle_dataset(slug):    direct = f'/kaggle/input/{slug}'    if os.path.isdir(direct):        return direct    candidates = glob.glob(f'/kaggle/input/datasets/*/{slug}')    return candidates[0] if candidates else Noneprint("🔍 Auto-detect Kaggle dataset paths:\n")mimic_data_root = _find_kaggle_dataset('forget-mi-data')mimic_models_root = _find_kaggle_dataset('forget-mi-models')print(f"📦 forget-mi-data   → {mimic_data_root or '❌ NOT FOUND'}")print(f"📦 forget-mi-models → {mimic_models_root or '❌ NOT FOUND'}\n")KAGGLE_MIMIC_DATA_ROOT = mimic_data_rootKAGGLE_MIMIC_MODELS_ROOT = mimic_models_rootif mimic_data_root and mimic_models_root:    checks = {        "Base model":      f"{mimic_models_root}/base_model/training_original_model/pytorch_model.bin",        "Retrained 3%":    f"{mimic_models_root}/retrained_model/model_retrained_3per/pytorch_model.bin",        "Text metadata":   f"{mimic_data_root}/data/metadata",        "Image data":      f"{mimic_data_root}/data/img_data",        "MIMIC split CSV": "./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv",        "Synonyms":        "./data_splits/Synonyms.csv",        "Forget 3% list":  "./data_splits/forget_set_3per.csv",    }    all_ok = True    for name, path in checks.items():        exists = os.path.exists(path)        print(f"  {'✅' if exists else '❌'} {name:<18} {path}")        if not exists:            all_ok = False    print()    if all_ok:        print("✅ Tất cả paths OK — sẵn sàng chạy LoKU")    else:        print("⚠️  Thiếu paths — kiểm tra Kaggle Datasets đã add đúng chưa")else:    print("❌ Thiếu Kaggle Dataset. Thêm vào notebook Sidebar → + Add data:")    print("   - forget-mi-data")    print("   - forget-mi-models")

In [ ]:
# ====================================# CELL 3: Helpers cho LoKU multi-seed (BẮT BUỘC trước Cell 4*)# ====================================import os, numpy as np, pandas as pdfrom datetime import datetimeCSV_PATH = "/kaggle/working/unlearning_output/results_summary.csv"PAPER_REF = {    3:  {"MIA_paper": 0.571, "Df_AUC": 0.735, "Df_F1": 0.393, "Dt_AUC": 0.625, "Dt_F1": 0.250, "Time_h": 5.0},    6:  {"MIA_paper": 0.615, "Df_AUC": 0.654, "Df_F1": 0.328, "Dt_AUC": 0.599, "Dt_F1": 0.270, "Time_h": 5.0},    10: {"MIA_paper": 0.810, "Df_AUC": 0.656, "Df_F1": 0.313, "Dt_AUC": 0.565, "Dt_F1": 0.252, "Time_h": 5.0},}def _gold_path(forget_pct):    if forget_pct != 3 or not KAGGLE_MIMIC_MODELS_ROOT:        return None    p = f"{KAGGLE_MIMIC_MODELS_ROOT}/retrained_model/model_retrained_3per"    return p if os.path.isdir(p) else NoneMETRIC_DEFS = [    ('MIA',                 'MIA_persample',   None,         '↓'),    ('MIA_paper',           'MIA_paper',       'MIA_paper',  '↓'),    ('forget_ce',           'forget_ce',       None,         '·'),    ('test_ce',             'test_ce',         None,         '·'),    ('Df_AUC',              'Forget AUC',      'Df_AUC',     '↓'),    ('Df_F1',               'Forget Mac-F1',   'Df_F1',      '↓'),    ('Dt_AUC',              'Test AUC',        'Dt_AUC',     '↑'),    ('Dt_F1',               'Test Mac-F1',     'Dt_F1',      '↑'),    ('dist_vs_re',          '1 − CosSim',      None,         '↓'),    ('unlearn_time_hours',  'Time (h)',        'Time_h',     '↓'),    ('gpu_peak_GB',         'GPU peak (GB)',   None,         '·'),    ('trainable_ratio',     'Trainable ratio', None,         '↓'),]def _seed_done(forget_pct, seed):    if not os.path.exists(CSV_PATH):        return False    try:        df = pd.read_csv(CSV_PATH)    except Exception:        return False    if 'forget_pct' not in df.columns or 'seed' not in df.columns:        return False    mask = df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per") & (df['seed'] == seed)    return bool(mask.any())def run_multiseed(forget_pct, seeds=(42, 123, 7), ihl=0.75, force_redo=False):    if isinstance(seeds, int):        seeds = (seeds,)    seeds = tuple(seeds)    assert forget_pct in (3, 6, 10)    forget_csv = f"./data_splits/forget_set_{forget_pct}per.csv"    gold_path = _gold_path(forget_pct)    has_gold = gold_path is not None    paper = PAPER_REF[forget_pct]    exp_name = f"final_ihl{int(ihl*100):03d}_{forget_pct}per"    print(f"\n{'#'*72}")    print(f"# 🎯 LoKU — FORGET {forget_pct}% — seeds={list(seeds)} — IHL={ihl}")    print(f"# Forget CSV    : {forget_csv}")    if has_gold:        print(f"# Gold retrained: ✅ {gold_path}")    else:        print(f"# Gold retrained: ❌ N/A (1−CosSim KHÔNG hợp lệ)")    print(f"{'#'*72}\n")    if not os.path.exists(forget_csv):        raise FileNotFoundError(f"Forget set không tồn tại: {forget_csv}")    OVR = (f"forget_set_path={forget_csv},id=loku_{forget_pct}per,ihl_forget_weight={ihl},"           f"base_model_path={KAGGLE_MIMIC_MODELS_ROOT}/base_model/training_original_model,"           f"bert_pretrained_dir={KAGGLE_MIMIC_MODELS_ROOT}/base_model/training_original_model,"           f"text_data_dir={KAGGLE_MIMIC_DATA_ROOT}/data/metadata,"           f"img_data_dir={KAGGLE_MIMIC_DATA_ROOT}/data/img_data")    if has_gold:        OVR += f",retrained_model_path={gold_path}"    HYP = (f"LoKU Kaggle FORGET {forget_pct}%: distill_teacher=og, early_stop=val, "           f"IHL={ihl}, image-FILA scale0.3. Multi-seed mean+-std.")    for i, s in enumerate(seeds):        if not force_redo and _seed_done(forget_pct, s):            print(f"⏭️  SEED {s} đã có trong CSV — skip (force_redo=True để rerun)\n")            continue        print(f"\n{'='*60}\n🎲 SEED {s} ({i+1}/{len(seeds)}) ▸ LoKU FORGET {forget_pct}%\n{'='*60}")        cmd = (f'PYTHONPATH=. WANDB_MODE=disabled python training/forgetmi_loku.py '               f'--config config_loku_kaggle.yaml --fresh --seed {s} '               f'--override "{OVR}" '               f'--exp {exp_name}_seed{s} --hypothesis "{HYP}"')        get_ipython().system(cmd)    aggregate_summary(forget_pct, seeds, exp_name, paper, has_gold, ihl=ihl)def aggregate_summary(forget_pct, seeds, exp_name, paper_ref, has_gold, ihl=0.75):    if not os.path.exists(CSV_PATH):        print(f"❌ Không tìm thấy {CSV_PATH}. Bỏ qua aggregate.")        return    df = pd.read_csv(CSV_PATH)    mask = df['forget_pct'].astype(str).str.contains(f"_{forget_pct}per") & df['seed'].isin(seeds)    df = df[mask]    if df.empty:        print(f"❌ Không có rows cho {forget_pct}% + seeds={list(seeds)}.")        return    if 'timestamp' in df.columns:        df = df.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')    print(f"\n{'='*100}")    print(f"📊 LoKU MULTI-SEED — FORGET {forget_pct}%, seeds={list(seeds)}")    if not has_gold:        print(f"⚠️  Gold retrained N/A cho {forget_pct}% → 1−CosSim KHÔNG hợp lệ")    print('=' * 100)    hdr = (f"{'Metric':<20}" + "".join(f"{f'seed{s}':>10}" for s in seeds)           + f"{'mean ± std':>18}" + f"{'paper':>10}" + f"{'Δ vs paper':>12}")    print(hdr); print("-" * len(hdr))    md_rows = [        "| Metric | " + " | ".join(f"seed {s}" for s in seeds)        + " | **mean ± std** | Paper | Δ vs paper |",        "|---|" + "---|" * (len(seeds) + 3),    ]    for csv_key, label, paper_key, direction in METRIC_DEFS:        if csv_key not in df.columns:            continue        vals = []        for s in seeds:            sub = df[df['seed'] == s]            if sub.empty:                continue            try:                v = float(sub[csv_key].iloc[-1])            except Exception:                continue            if v != v:                continue            vals.append(v)        if not vals:            continue        m, sd = float(np.mean(vals)), float(np.std(vals))        inv = " ⚠️" if (csv_key == 'dist_vs_re' and not has_gold) else ""        if paper_key and paper_key in paper_ref:            p_val = paper_ref[paper_key]            delta = m - p_val            arrow_good = (direction == '↓' and delta < 0) or (direction == '↑' and delta > 0)            sym = "✅" if arrow_good else ("❌" if direction in ('↓', '↑') else "·")            paper_str = f"{p_val:>10.3f}"            delta_str = f"{sym}{delta:+.3f}"        else:            paper_str = f"{'—':>10}"            delta_str = "—"        cell_vals = "".join(f"{v:>10.3f}" for v in vals) + " " * (10 * (len(seeds) - len(vals)))        ms_str = f"{m:>10.3f}±{sd:.3f}"        print(f"{label+inv:<20}{cell_vals}{ms_str:>18}{paper_str}{delta_str:>12}")        md_vals = " | ".join(f"{v:.3f}" for v in vals) + " | " * (len(seeds) - len(vals))        md_paper = f"{paper_ref[paper_key]:.3f}" if paper_key and paper_key in paper_ref else "—"        md_delta = f"{delta:+.3f}" if paper_key and paper_key in paper_ref else "—"        md_rows.append(f"| {label}{inv} | {md_vals} | **{m:.3f} ± {sd:.3f}** | {md_paper} | {md_delta} |")    os.makedirs("experiments", exist_ok=True)    out_md = f"experiments/summary_{exp_name}_kaggle_multiseed.md"    body_lines = [        f"# LoKU Kaggle Multi-seed — FORGET {forget_pct}% — {exp_name}",        "",        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}_",        "",        f"**Config**: `final_ihl{int(ihl*100):03d}` (honest, no F_re), IHL={ihl}, image-FILA scale=0.3",        "",        "**Hardware**: Kaggle T4 (so sánh trực tiếp được với baseline run_kaggle_baseline.ipynb)",        "",        f"**Seeds**: {list(seeds)}",        "",        "**Gold retrained**: " + ("✅ available" if has_gold else "❌ N/A — 1−CosSim KHÔNG hợp lệ"),        "",    ]    body_lines.extend(md_rows)    body_lines.extend([        "",        f"**Paper Forget-MI ({forget_pct}%)**: MIA={paper_ref['MIA_paper']} | Df_AUC={paper_ref['Df_AUC']} | "        f"Dt_AUC={paper_ref['Dt_AUC']} | Time≈{paper_ref['Time_h']}h",        "",        "_Δ vs paper_: âm = LoKU tốt hơn (↓ metrics) hoặc kém hơn (↑ metrics). ✅ = LoKU thắng.",    ])    with open(out_md, "w", encoding="utf-8") as f:        f.write("\n".join(body_lines))    print(f"\n💾 Summary MD: {out_md}")print("✅ Helpers đã định nghĩa: run_multiseed(), aggregate_summary()")print(f"   CSV path: {CSV_PATH}")print("   Config  : config_loku_kaggle.yaml")

---## 🎯 Training cells — chạy theo nhu cầu| Cell | Forget% | Có RUN flag? | Mặc định ||---|---|---|---|| 4a | 3% | Không | ✅ Chạy || 4b | 6% | `RUN_6PER` | ❌ Skip || 4c | 10% | `RUN_10PER` | ❌ Skip |Save Version chạy unattended → chỉ Cell 4a active. Khi muốn thêm 6%/10% → mở cell tương ứng set `True` rồi Save Version mới.

In [ ]:
# ====================================# CELL 4a: LoKU MULTI-SEED FORGET 3% (gold ✅)# ====================================# Expected (Colab reference): MIA≈0.43, Df_AUC≈0.74, Dt_AUC≈0.68, Time≈0.2h# Trên T4 Kaggle có thể chậm hơn Colab nhẹ.SEEDS_THIS_RUN = (42, 123, 7)run_multiseed(forget_pct=3, seeds=SEEDS_THIS_RUN, ihl=0.75)

In [ ]:
# ====================================# CELL 4b: LoKU MULTI-SEED FORGET 6% (no gold)# ====================================RUN_6PER = False   # ⚙️ Set True khi muốn train 6%SEEDS_THIS_RUN = (42, 123, 7)if RUN_6PER:    run_multiseed(forget_pct=6, seeds=SEEDS_THIS_RUN, ihl=0.75)else:    print("⏭️  Skip LoKU 6% (RUN_6PER=False)")

In [ ]:
# ====================================# CELL 4c: LoKU MULTI-SEED FORGET 10% (no gold)# ====================================RUN_10PER = False   # ⚙️ Set True khi muốn train 10%SEEDS_THIS_RUN = (42, 123, 7)if RUN_10PER:    run_multiseed(forget_pct=10, seeds=SEEDS_THIS_RUN, ihl=0.75)else:    print("⏭️  Skip LoKU 10% (RUN_10PER=False)")

In [ ]:
# ====================================# CELL 5: Bảng cross-forget% — LoKU Kaggle# ====================================import os, pandas as pdfrom datetime import datetimeif not os.path.exists(CSV_PATH):    print(f"❌ {CSV_PATH} không tồn tại. Cần chạy ít nhất 1 Cell 4*.")else:    df_all = pd.read_csv(CSV_PATH)    pcts_in_csv = sorted({int(s.split('_')[-1].replace('per.csv', '').replace('per', ''))                          for s in df_all['forget_pct'].dropna().astype(str).unique() if '_' in s})    print(f"📋 Total rows: {len(df_all)} — forget% có data: {pcts_in_csv}\n")    show_metrics = [        ('MIA_paper', 'MIA', '↓'), ('Df_AUC', 'Df_AUC', '↓'), ('Df_F1', 'Df_F1', '↓'),        ('Dt_AUC', 'Dt_AUC', '↑'), ('Dt_F1', 'Dt_F1', '↑'),        ('dist_vs_re', '1−CosSim', '↓'), ('unlearn_time_hours', 'Time(h)', '↓'),    ]    md_lines = [        "# LoKU Kaggle — Bảng cross-forget%",        "",        f"_Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}_",        "",        "Kết quả LoKU multi-seed trên Kaggle T4 (so trực tiếp với baseline cùng GPU).",        "",    ]    header = "| Forget% | Method | " + " | ".join(m[1] for m in show_metrics) + " |"    sep = "|---|---|" + "---|" * len(show_metrics)    md_lines += [header, sep]    print("=" * 110); print(header); print("=" * 110)    paper_key_map = {'MIA_paper': 'MIA_paper', 'Df_AUC': 'Df_AUC', 'Df_F1': 'Df_F1',                     'Dt_AUC': 'Dt_AUC', 'Dt_F1': 'Dt_F1', 'unlearn_time_hours': 'Time_h'}    for pct in [3, 6, 10]:        sub = df_all[df_all['forget_pct'].astype(str).str.contains(f"_{pct}per")]        if sub.empty:            row = f"| {pct}% | _(chưa chạy)_ |" + " — |" * len(show_metrics)            md_lines.append(row); print(row); continue        if 'timestamp' in sub.columns:            sub = sub.sort_values('timestamp').drop_duplicates(subset=['seed'], keep='last')        n = sub['seed'].nunique()        # Paper row        paper = PAPER_REF.get(pct, {})        paper_row = f"| {pct}% | Paper Forget-MI |"        for csv_k, _, _ in show_metrics:            pk = paper_key_map.get(csv_k)            paper_row += f" {paper.get(pk, '—')} |" if pk and pk in paper else " — |"        md_lines.append(paper_row); print(paper_row)        # LoKU row (mean of seeds)        loku_row = f"| {pct}% | LoKU (n={n}) |"        for csv_k, _, _ in show_metrics:            if csv_k in sub.columns:                v = sub[csv_k].mean()                loku_row += f" {v:.3f} |" if pd.notna(v) else " — |"            else:                loku_row += " — |"        md_lines.append(loku_row); print(loku_row)        md_lines.append("")    out_md = "experiments/bang_loku_kaggle_cross_forget.md"    os.makedirs("experiments", exist_ok=True)    with open(out_md, "w", encoding="utf-8") as f:        f.write("\n".join(md_lines))    print(f"\n💾 Saved: {out_md}")

In [ ]:
# ====================================# CELL 6: Push LoKU results lên GitHub (Kaggle Secrets)# ====================================import os, shutilGITHUB_REPO = "nhnhu146/Forget-MI-LoKU"BRANCH = "master"# BƯỚC 1: Copy CSVCSV_SRC = "/kaggle/working/unlearning_output/results_summary.csv"CSV_DST = "experiments/results_summary_loku_kaggle.csv"if os.path.exists(CSV_SRC):    os.makedirs("experiments", exist_ok=True)    shutil.copy(CSV_SRC, CSV_DST)    n = sum(1 for _ in open(CSV_DST)) - 1    print(f"📋 Copied CSV ({n} rows) → {CSV_DST}")else:    print(f"ℹ️  CSV chưa có ({CSV_SRC}) — chỉ push MD files")# BƯỚC 2: Load credentials từ Kaggle Secretsdef load_secrets():    try:        from kaggle_secrets import UserSecretsClient        secrets = UserSecretsClient()        return (secrets.get_secret('GITHUB_TOKEN'),                secrets.get_secret('GIT_EMAIL'),                secrets.get_secret('GIT_NAME'))    except Exception as e:        print(f"⚠️  Không load được Kaggle Secrets ({e})")        return None, None, NoneTOKEN, EMAIL, NAME = load_secrets()if TOKEN and EMAIL and NAME:    print("🔑 Credentials từ Kaggle Secrets ✅")else:    print("⚠️  Thiếu credentials — skip push")    raise SystemExit# BƯỚC 3: Git commit + pushget_ipython().system(f'git config user.email "{EMAIL}"')get_ipython().system(f'git config user.name "{NAME}"')get_ipython().system(f'git remote set-url origin https://{TOKEN}@github.com/{GITHUB_REPO}.git')get_ipython().system('git add experiments/ 2>/dev/null')changes = get_ipython().getoutput('git diff --cached --name-only')if changes and any(c.strip() for c in changes):    print("\n📦 Files commit:")    for f in changes:        if f.strip():            print(f"   - {f}")    msg = "loku kaggle: multi-seed results + CSV checkpoint"    get_ipython().system(f'git commit -m "{msg}"')    get_ipython().system(f'git push origin {BRANCH}')    print(f"\n✅ Pushed: https://github.com/{GITHUB_REPO}/tree/{BRANCH}/experiments")else:    print("ℹ️  Không có file mới để commit.")